In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from sklearn.metrics import classification_report

from src.functions.data_loader import load_raw, get_labeled

features, classes, edges = load_raw()
df = features.merge(classes, on="txId", how="left")

print(features.shape, classes.shape, edges.shape)
print("Labeled rows:", len(get_labeled(features, classes)))

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


(203769, 167) (203769, 2) (234355, 2)
Labeled rows: 46564


In [2]:
id_map = {tx: i for i, tx in enumerate(df["txId"])}

src = edges["txId1"].map(id_map).values
dst = edges["txId2"].map(id_map).values
edge_index = to_undirected(torch.tensor(np.array([src, dst]), dtype=torch.long))

feat_cols = [c for c in df.columns if c.startswith("feature_")]
x = torch.tensor(df[feat_cols].values, dtype=torch.float)

y = torch.tensor(df["class"].map({"1": 1, "2": 0}).fillna(-1).astype(int).values)
time = torch.tensor(df["time_step"].values)

train_mask = (y >= 0) & (time <= 34)
test_mask = (y >= 0) & (time > 34)

data = Data(x=x, edge_index=edge_index, y=y)
print(data)
print("Features used:", len(feat_cols))
print("Train nodes:", train_mask.sum().item(), "Test nodes:", test_mask.sum().item())
print("Test illicit:", (y[test_mask] == 1).sum().item())

Data(x=[203769, 165], edge_index=[2, 468710], y=[203769])
Features used: 165
Train nodes: 29894 Test nodes: 16670
Test illicit: 1083


In [3]:
torch.manual_seed(42)

class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=2):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)

model_gnn = GCN(x.shape[1])
optimizer = torch.optim.Adam(model_gnn.parameters(), lr=0.01, weight_decay=5e-4)

# Weight the rare illicit class more heavily
n_pos = (y[train_mask] == 1).sum().item()
n_neg = (y[train_mask] == 0).sum().item()
weights = torch.tensor([1.0, n_neg / n_pos])
print("Train illicit:", n_pos, "Train licit:", n_neg)
print("Illicit class weight:", round(weights[1].item(), 2))

Train illicit: 3462 Train licit: 26432
Illicit class weight: 7.63


In [4]:
for epoch in range(200):
    model_gnn.train()
    optimizer.zero_grad()
    out = model_gnn(data.x, data.edge_index)
    loss = F.cross_entropy(out[train_mask], data.y[train_mask], weight=weights)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, loss {loss.item():.4f}")

model_gnn.eval()
with torch.no_grad():
    pred = model_gnn(data.x, data.edge_index).argmax(dim=1)

print(classification_report(
    data.y[test_mask].numpy(), pred[test_mask].numpy(),
    target_names=["Licit", "Illicit"]))

Epoch 0, loss 0.8431
Epoch 20, loss 0.2549
Epoch 40, loss 0.2081
Epoch 60, loss 0.1778
Epoch 80, loss 0.1564
Epoch 100, loss 0.1437
Epoch 120, loss 0.1347
Epoch 140, loss 0.1274
Epoch 160, loss 0.1205
Epoch 180, loss 0.1144
              precision    recall  f1-score   support

       Licit       0.97      0.97      0.97     15587
     Illicit       0.62      0.63      0.63      1083

    accuracy                           0.95     16670
   macro avg       0.80      0.80      0.80     16670
weighted avg       0.95      0.95      0.95     16670

